In [12]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.tree import DecisionTreeRegressor

## 12. How far ahead can dissolved oxygen be predicted before the model stops beating a naive persistence baseline?

1. Data preparation
- 1. Considering data gaps, we use data in previous time slots to fill in the gaps. If there is no, we backtrack util there is one. 
- 2. Considering data, target, DO, changes based on time, we add previous DO, as features as refrences to assume they have correlcations.
- 3. Since the feature, temperature, is highly correlated to DO, we also previous temperature as features as refrences.

In [ ]:
CSV_PATH = "Test_Station.csv"
OUT_DIR = "results"
HORIZONS = {"1h": 4, "3h": 12, "6h": 24, "12h": 48, "24h": 96}  # 15-min rows per horizon
HYPOXIA = 2.0          # mg/L threshold
TRAIN_FRAC = 0.70      # first 70% of rows train, last 30% test
RANDOM_STATE = 0

In [14]:
def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))


def mae(a, b):
    return float(np.mean(np.abs(a - b)))

Parse the raw measurement columns, check time coverage and missingness, and apply the existing nonnegative clipping rule for DO and saturation.

In [15]:
os.makedirs(OUT_DIR, exist_ok=True)
raw = pd.read_csv(CSV_PATH)

# Column names as they appear in the raw 'Continuous Measurement' download
df = pd.DataFrame({
    "date": pd.to_datetime(raw["DateTime"], format="%m/%d/%Y %H:%M", errors="coerce"),
    "do": pd.to_numeric(raw["DO_mg/L"], errors="coerce"),
    "do_sat": pd.to_numeric(raw["DO_pctSat"], errors="coerce"),
    "sal": pd.to_numeric(raw["Salinity_ppt"], errors="coerce"),
    "ph": pd.to_numeric(raw["pH"], errors="coerce"),
    "turb": pd.to_numeric(raw["Turb_NTU"], errors="coerce"),
    "chl": pd.to_numeric(raw["Chl_ug/L"], errors="coerce"),
    "temp": pd.to_numeric(raw["Temp_C"], errors="coerce"),
})
n = len(df)

checks = {"rows": n, "rows_with_do": int(df["do"].notna().sum()),
          "expected_rows_apr1_oct1_2024": 17664}
valid_dates = df["date"].notna().mean()
checks["valid_date_share"] = round(float(valid_dates), 4)
if valid_dates >= 0.99:
    steps = df["date"].diff().dropna()
    checks["gaps_not_15min"] = int((steps != pd.Timedelta(minutes=15)).sum())
    checks["date_min"] = str(df["date"].min())
    checks["date_max"] = str(df["date"].max())
    if checks["gaps_not_15min"] > 0:
        print("WARNING: some rows are not exactly 15 minutes apart. Horizons are row shifts, "
              "so they will be approximate around those gaps.")
else:
    print("WARNING: Date column is not usable. Row order will be treated as time.")

if n != checks["expected_rows_apr1_oct1_2024"]:
    print(f"NOTE: {n} rows, README expects 17664. Check the download date range.")

# EDA checks that the README asks for
checks["do_negative"] = int((df["do"] < 0).sum())
checks["do_hypoxic_lt2"] = int((df["do"] < HYPOXIA).sum())
checks["missing_pct"] = {c: round(100 * df[c].isna().mean(), 2)
                         for c in ["sal", "ph", "do", "do_sat", "turb", "chl", "temp"]}
checks["corr_do_sat_vs_do"] = round(float(df["do"].corr(df["do_sat"])), 3)
checks["corr_sat_vs_ph"] = round(float(df["do_sat"].corr(df["ph"])), 3)
checks["acf_6h"] = round(float(df["do"].autocorr(24)), 3)
checks["acf_12h"] = round(float(df["do"].autocorr(48)), 3)
checks["acf_24h"] = round(float(df["do"].autocorr(96)), 3)

# README: keep rows with missing values; clip small negatives to zero
df["do"] = df["do"].clip(lower=0)
df["do_sat"] = df["do_sat"].clip(lower=0)

FileNotFoundError: [Errno 2] No such file or directory: './Station_Transfer/AEB.csv'

## Build forecasting features

Features use current and lagged measurements only. The chronological split keeps the final 30% of rows for evaluation.

In [ ]:
X = pd.DataFrame(index=df.index)
for c in ["do", "sal", "ph", "turb", "chl", "temp"]:
    X[c] = df[c]
for lag in [1, 2, 4, 8, 16, 32]:
    X[f"do_lag{lag}"] = df["do"].shift(lag)
for c in ["sal", "ph", "turb", "chl", "temp"]:
    for lag in [4, 16]:
        X[f"{c}_lag{lag}"] = df[c].shift(lag)
X["do_diff1"] = df["do"] - df["do"].shift(1)
X["do_diff4"] = df["do"] - df["do"].shift(4)
X["temp_diff4"] = df["temp"] - df["temp"].shift(4)
X["do_roll16_mean"] = df["do"].rolling(16, min_periods=8).mean()
X["do_roll96_mean"] = df["do"].rolling(96, min_periods=48).mean()
X = X.astype(float).values

current = df["do"].values
idx = np.arange(n)
split = int(TRAIN_FRAC * n)


2. Models that we go to use and expirements that will beconducted.
- 1. Even though the data are time series, some data points are missed,
So, I will mainly focus on other models than RNN.  
- 2. Since problem is based a predicating values are continious, we use regression models and compete the performance.
 the regression models are Linear regression, Ridge regression, Quadratic Ridge, Decision tree, Random forest, Neural network. 
- 3. Consider, it is time series data with big gaps, we could consider Histogram gradient boosting. 
- 4. Since the problem is time sentive, we could like to try use models to predict DO in advances for different time ranges.

In [ ]:
def make_models():
    return {
        "linear": make_pipeline(
            SimpleImputer(strategy="median"), StandardScaler(), LinearRegression()
        ),
        "ridge": make_pipeline(
            SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=1.0)
        ),
        "quadratic_ridge": make_pipeline(
            SimpleImputer(strategy="median"),
            PolynomialFeatures(degree=2, include_bias=False),
            StandardScaler(),
            Ridge(alpha=1.0),
        ),
        "decision_tree": make_pipeline(
            SimpleImputer(strategy="median"),
            DecisionTreeRegressor(random_state=RANDOM_STATE),
        ),
        "random_forest": make_pipeline(
            SimpleImputer(strategy="median"),
            RandomForestRegressor(
                n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
            ),
        ),
        "neural_network": make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(),
            MLPRegressor(
                hidden_layer_sizes=(64, 32), max_iter=500, random_state=RANDOM_STATE
            ),
        ),
        "hist_gbm": HistGradientBoostingRegressor(
            max_iter=300, learning_rate=0.05, random_state=RANDOM_STATE
        ),
    }

b. Traing and testing

In [ ]:
# preparing the data
results = []
hypoxia_rows = []
plot_store = None

for hname, h in HORIZONS.items():
    y = df["do"].shift(-h).values
    valid = ~np.isnan(y) & ~np.isnan(current)
    tr = (idx < split - h) & valid      # drop training targets that reach into the test period
    te = (idx >= split) & valid
    y_tr = y[tr]
    y_te = y[te]

    preds = {"persistence": current[te]}
    train_preds = {"persistence": current[tr]}
    for name, model in make_models().items():
        model.fit(X[tr], (y - current)[tr])          # learn the change, add it back
        train_preds[name] = current[tr] + model.predict(X[tr])
        preds[name] = current[te] + model.predict(X[te])

    base = rmse(y_te, preds["persistence"])
    train_base = rmse(y_tr, train_preds["persistence"])
    row = {"horizon": hname, "n_train": int(tr.sum()), "n_test": int(te.sum()), "train": {}}
    for name, p in train_preds.items():
        r = rmse(y_tr, p)
        row["train"][name] = {
            "rmse": round(r, 4),
            "mae": round(mae(y_tr, p), 4),
            "improvement_vs_persistence_pct": round(100 * (train_base - r) / train_base, 1),
            "meets_20pct_target": bool(r <= 0.8 * train_base),
        }
    for name, p in preds.items():
        r = rmse(y_te, p)
        row[name] = {
            "rmse": round(r, 4),
            "mae": round(mae(y_te, p), 4),
            "improvement_vs_persistence_pct": round(100 * (base - r) / base, 1),
            "meets_20pct_target": bool(r <= 0.8 * base),
        }
    results.append(row)

    actual_hyp_train = (y_tr < HYPOXIA).astype(int)
    for name, p in train_preds.items():
        pred_hyp_train = (p < HYPOXIA).astype(int)
        hypoxia_rows.append({
            "horizon": hname, "split": "train", "model": name,
            "precision": round(precision_score(actual_hyp_train, pred_hyp_train, zero_division=0), 3),
            "recall": round(recall_score(actual_hyp_train, pred_hyp_train, zero_division=0), 3),
            "f1": round(f1_score(actual_hyp_train, pred_hyp_train, zero_division=0), 3),
        })
    actual_hyp = (y_te < HYPOXIA).astype(int)
    for name, p in preds.items():
        pred_hyp = (p < HYPOXIA).astype(int)
        hypoxia_rows.append({
            "horizon": hname, "split": "test", "model": name,
            "precision": round(precision_score(actual_hyp, pred_hyp, zero_division=0), 3),
            "recall": round(recall_score(actual_hyp, pred_hyp, zero_division=0), 3),
            "f1": round(f1_score(actual_hyp, pred_hyp, zero_division=0), 3),
        })

    if hname == "6h":
        plot_store = (y_te, preds)

c. Performance measurements

Training scores are in-sample and can look optimistic; test scores use held-out future rows. The plots compare RMSE, MAE, precision, recall, and F1 across forecast horizons.

In [ ]:
score_rows = []
for result in results:
    test_metrics = {
        name: scores for name, scores in result.items()
        if name not in {"horizon", "n_train", "n_test", "train"}
    }
    for split_name, metrics_by_model in (
        ("train", result["train"]),
        ("test", test_metrics),
    ):
        for model_name, scores in metrics_by_model.items():
            score_rows.append({
                "horizon": result["horizon"],
                "split": split_name,
                "model": model_name,
                **scores,
            })

scores_df = pd.DataFrame(score_rows)
hypoxia_scores_df = pd.DataFrame(hypoxia_rows)
display(scores_df.sort_values(["horizon", "model", "split"]))
display(hypoxia_scores_df.sort_values(["horizon", "model", "split"]))

horizon_order = list(HORIZONS)
model_order = list(dict.fromkeys(scores_df["model"]))
plot_specs = [
    ("rmse", scores_df, "RMSE (mg/L)", None),
    ("mae", scores_df, "MAE (mg/L)", None),
    ("precision", hypoxia_scores_df, "Precision", (0, 1)),
    ("recall", hypoxia_scores_df, "Recall", (0, 1)),
    ("f1", hypoxia_scores_df, "F1 score", (0, 1)),
]
fig, axes = plt.subplots(len(plot_specs), 2, figsize=(18, 22), sharex=True)
for row_index, (metric, data, ylabel, ylim) in enumerate(plot_specs):
    for col_index, split_name in enumerate(["train", "test"]):
        ax = axes[row_index, col_index]
        for model_name in model_order:
            subset = data[
                (data["split"] == split_name) & (data["model"] == model_name)
            ]
            ax.plot(subset["horizon"], subset[metric], marker="o", label=model_name)
        ax.set_ylabel(ylabel)
        if ylim is not None:
            ax.set_ylim(*ylim)
        if row_index == 0:
            ax.set_title(f"{split_name.title()} scores")
        if row_index == 0 and col_index == 0:
            ax.legend(fontsize=8)
        if row_index == len(plot_specs) - 1:
            ax.set_xlabel("Forecast horizon")
            ax.set_xticks(range(len(horizon_order)))
            ax.set_xticklabels(horizon_order)
fig.suptitle("Training and test scores by forecast horizon")
fig.tight_layout()
plt.show()

d. Conclusion

1. generally Overfit in all models.
2. Less than 6 hour predict beats persistence. 